### Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

1.Найти велосипед с максимальным временем пробега.

2.Найти наибольшее геодезическое расстояние между станциями.

3.Найти путь велосипеда с максимальным временем пробега через станции.

4.Найти количество велосипедов в системе.

5.Найти пользователей потративших на поездки более 3 часов.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("L1_Apache_Spark")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

def find_file(filename: str) -> Path:
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Не найден файл {filename}. Проверены пути: " +
        ", ".join(str(path) for path in candidates)
    )

trips_csv = find_file("trips.csv")
stations_csv = find_file("stations.csv")

trip_raw = spark.read.option("header", True).csv(str(trips_csv))
stations_raw = spark.read.option("header", True).csv(str(stations_csv))

trips_df = (
    trip_raw
    .select(
        F.col("id").cast("int").alias("id"),
        F.col("duration").cast("int").alias("duration"),
        F.coalesce(
            F.to_timestamp("start_date", "M/d/yyyy H:mm"),
            F.to_timestamp("start_date", "M/d/yyyy H:mm:ss")
        ).alias("start_date"),
        F.col("start_station_name").alias("start_station_name"),
        F.col("start_station_id").cast("int").alias("start_station_id"),
        F.coalesce(
            F.to_timestamp("end_date", "M/d/yyyy H:mm"),
            F.to_timestamp("end_date", "M/d/yyyy H:mm:ss")
        ).alias("end_date"),
        F.col("end_station_name").alias("end_station_name"),
        F.col("end_station_id").cast("int").alias("end_station_id"),
        F.col("bike_id").cast("int").alias("bike_id"),
        F.trim(F.col("subscription_type")).alias("subscription_type"),
        F.trim(F.col("zip_code")).alias("zip_code"),
    )
)

stations_df = (
    stations_raw
    .select(
        F.col("id").cast("int").alias("id"),
        F.col("name").alias("name"),
        F.col("lat").cast("double").alias("lat"),
        F.col("long").cast("double").alias("long"),
        F.col("dock_count").cast("int").alias("dock_count"),
        F.col("city").alias("city"),
        F.coalesce(
            F.to_date("installation_date", "M/d/yyyy"),
            F.to_date("installation_date", "yyyy-MM-dd")
        ).alias("installation_date"),
    )
)

trips_df.printSchema()
stations_df.printSchema()

trips_df = (
    trips_df
    .where(F.col("id").isNotNull())
    .where(F.col("duration").isNotNull())
    .where(F.col("bike_id").isNotNull())
    .cache()
)

stations_df = (
    stations_df
    .where(F.col("id").isNotNull())
    .where(F.col("lat").isNotNull())
    .where(F.col("long").isNotNull())
    .cache()
)

print(f"Количество строк в trips_df: ", trips_df.count())
print(f"Количество строк в stations_df: ", stations_df.count())

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: date (nullable = true)

Количество строк в trips_df:  669958
Количество строк в stations_df:  70


### 1. Найти велосипед с максимальным временем пробега.

In [3]:
bike_time_stats = (
    trips_df
    .groupBy("bike_id")
    .agg(
        F.sum("duration").alias("ride_time_sec")
    )
    .sort(F.col("ride_time_sec").desc(), F.col("bike_id").asc())
)

max_bike = bike_time_stats.collect()[0]

bike_id_max_time = max_bike["bike_id"]
total_time_sec = max_bike["ride_time_sec"]
total_time_hours = total_time_sec / 3600.0

print(f"ID велосипеда с максимальным временем пробега: {bike_id_max_time}")
print(f"Общее время поездок: {total_time_sec} сек.")
print(f"Общее время поездок в часах: {total_time_hours:.2f}")

ID велосипеда с максимальным временем пробега: 535
Общее время поездок: 18611693 сек.
Общее время поездок в часах: 5169.91


### 2. Найти наибольшее геодезическое расстояние между станциями.

In [4]:
EARTH_RADIUS_KM = 6371

stations_left = stations_df.select(
    F.col("id").alias("station_id_left"),
    F.col("name").alias("station_name_left"),
    F.col("city").alias("station_city_left"),
    F.radians(F.col("lat")).alias("lat_left"),
    F.radians(F.col("long")).alias("lon_left")
)

stations_right = stations_df.select(
    F.col("id").alias("station_id_right"),
    F.col("name").alias("station_name_right"),
    F.col("city").alias("station_city_right"),
    F.radians(F.col("lat")).alias("lat_right"),
    F.radians(F.col("long")).alias("lon_right")
)

lat_diff = F.col("lat_right") - F.col("lat_left")
lon_diff = F.col("lon_right") - F.col("lon_left")

haversine_part = (
    F.pow(F.sin(lat_diff / 2), 2)
    + F.cos(F.col("lat_left")) * F.cos(F.col("lat_right")) * F.pow(F.sin(lon_diff / 2), 2)
)

distance_km_expr = 2 * F.lit(EARTH_RADIUS_KM) * F.asin(F.sqrt(haversine_part))

farthest_pair = (
    stations_left
    .crossJoin(stations_right)
    .filter(F.col("station_id_left") < F.col("station_id_right"))
    .withColumn("geo_distance_km", distance_km_expr)
    .sort(F.col("geo_distance_km").desc())
    .first()
)

print("Пара станций с наибольшим расстоянием:")
print(f"1) {farthest_pair['station_name_left']} ({farthest_pair['station_city_left']})")
print(f"2) {farthest_pair['station_name_right']} ({farthest_pair['station_city_right']})")
print(f"Расстояние между станциями: {farthest_pair['geo_distance_km']:.2f} км")

Пара станций с наибольшим расстоянием:
1) SJSU - San Salvador at 9th (San Jose)
2) Embarcadero at Sansome (San Francisco)
Расстояние между станциями: 69.92 км


### 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [11]:
bike_trips_df = (
    trips_df
    .filter(F.col("bike_id") == bike_id_max_time)
    .select(
        F.col("id"),
        F.col("start_date"),
        F.col("start_station_name"),
        F.col("end_date"),
        F.col("end_station_name"),
        F.col("duration")
    )
    .sort("start_date", "end_date", "id")
)

trip_total = bike_trips_df.count()
print(f"Всего поездок: {trip_total}")

path_data = bike_trips_df.select("start_station_name", "end_station_name").collect()
full_route = [path_data[0]["start_station_name"]]
full_route.extend(item["end_station_name"] for item in path_data)

print("Путь через станции:")
print(" -> ".join(full_route))

Всего поездок: 1328
Путь через станции:
Post at Kearney -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome -> 2nd at South Park -> Davis at Jackson -> Civic Center BART (7th at Market) -> Post at Kearney -> Embarcadero at Sansome -> Washington at Kearney -> Market at Sansome -> Market at Sansome -> 2nd at Folsom -> 2nd at Townsend -> 2nd at Townsend -> Embarcadero at Sansome -> Clay at Battery -> Harry Bridges Plaza (Ferry Building) -> Clay at Battery -> San Francisco Caltrain (Townsend at 4th) -> Steuart at Market -> 2nd at Townsend -> Harry Bridges Plaza (Ferry Building) -> Townsend at 7th -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Townsend at 7th -> San Francisco Caltrain (Townsend at 4th) -> 2nd at South Park -> 5th at Howard -> San Francisco Caltrain 2 (330 Townsend) -> Mechanics Plaza (Market at Battery) -> Powell at Post (Union Square) -> Powell at Post (Union Square) -> 5th a

### 4. Найти количество велосипедов в системе.

In [12]:
bike_count = trips_df.select("bike_id").distinct().count()

print(f"Количество велосипедов в системе: {bike_count}")

Количество велосипедов в системе: 700


### 5. Найти пользователей потративших на поездки более 3 часов.

In [15]:
users_over_3h_df = (
    trips_df
    .filter(F.col("zip_code").isNotNull())
    .filter(F.trim(F.col("zip_code")) != "")
    .groupBy("zip_code")
    .agg(
        F.sum(F.col("duration")).alias("sum_duration_sec")
    )
    .filter(F.col("sum_duration_sec") > 3 * 3600)
    .withColumn(
        "sum_duration_hours",
        F.round(F.col("sum_duration_sec") / 3600.0, 2)
    )
    .sort(F.col("sum_duration_sec").desc(), F.col("zip_code").asc())
)

print("Пользователи, потратившие на поездки более 3 часов: ")

users_count = users_over_3h_df.count()

users_over_3h_df.show(users_count ,truncate=False)

Пользователи, потратившие на поездки более 3 часов: 
+----------+----------------+------------------+
|zip_code  |sum_duration_sec|sum_duration_hours|
+----------+----------------+------------------+
|94107     |49757162        |13821.43          |
|nil       |45725550        |12701.54          |
|94105     |25596128        |7110.04           |
|94133     |21637675        |6010.47           |
|94102     |19128021        |5313.34           |
|94103     |19127388        |5313.16           |
|95531     |17270400        |4797.33           |
|94111     |14244997        |3956.94           |
|95112     |12742370        |3539.55           |
|94109     |12057128        |3349.2            |
|94040     |7807926         |2168.87           |
|94110     |7421936         |2061.65           |
|94117     |6901313         |1917.03           |
|94301     |6590378         |1830.66           |
|94041     |6276284         |1743.41           |
|94158     |6248167         |1735.6            |
|94306     |5550

In [16]:
spark.stop()